# EDA MovieLens avec le code du projet

Ce notebook est une couche de démonstration. La logique métier vient des modules du projet.

## 0) Imports et création Spark

In [ ]:
from pathlib import Path
import sys
import pandas as pd
import seaborn as sns
import matplotlib.pyplot as plt

ROOT = Path.cwd()
if not (ROOT / 'src').exists():
    ROOT = ROOT.parent

if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from src.config.settings import PipelineSettings
from src.preprocessing.spark_session import create_spark
from src.ingestion.load_data import load_all_data
from src.preprocessing.clean_data import clean_movies, clean_tags, clean_ratings, time_based_split
from src.preprocessing.feature_engineering import add_time_features, encode_genres, build_movie_genre_weights, build_tag_features
from src.preprocessing.user_profiles import create_user_profiles, build_user_genre_profiles, build_user_tag_profiles
from src.models.als_model import train_als_with_tuning, retrain_best_als, score_als, recommend_for_all_users_flat, score_als_candidates
from src.models.content_model import generate_content_candidates, build_content_scores
from src.models.hybrid_model import combine_hybrid_scores, select_top_k_recommendations
from src.evaluation.rmse import compute_rmse
from src.evaluation.mae import compute_mae
from src.evaluation.precision_at_k import compute_precision_at_k
from src.pipelines.training_pipeline import run_pipeline

sns.set_theme(style='whitegrid', context='talk')
plt.rcParams['figure.figsize'] = (12, 6)

settings = PipelineSettings()
spark = create_spark(settings)
spark

## 1) Chargement des données (module ingestion)

In [ ]:
ratings_raw, movies_raw, tags_raw = load_all_data(spark=spark, settings=settings, prefer_parquet=False)
ratings_raw.printSchema()
movies_raw.printSchema()
tags_raw.printSchema()

## 2) Prétraitement (modules preprocessing)

In [ ]:
movies_clean = clean_movies(movies_raw).cache()
tags_clean = clean_tags(tags_raw).cache()
ratings_clean = clean_ratings(ratings_raw, settings=settings).cache()
ratings_feat = add_time_features(ratings_clean).cache()

movie_genres = encode_genres(movies_clean).cache()
movie_genre_weights = build_movie_genre_weights(movie_genres).cache()
user_profiles = create_user_profiles(ratings_feat).cache()

train_df, val_df, test_df = time_based_split(ratings_feat, settings=settings)
train_df = train_df.cache()
val_df = val_df.cache()
test_df = test_df.cache()

print('Train:', train_df.count(), 'Validation:', val_df.count(), 'Test:', test_df.count())

## 3) Vue globale des données

In [ ]:
n_users = ratings_clean.select('userId').distinct().count()
n_movies = movies_clean.select('movieId').distinct().count()
n_ratings = ratings_clean.count()
sparsity = 1 - (n_ratings / (n_users * n_movies))

pd.DataFrame({
    'Métrique': ['Nombre utilisateurs', 'Nombre films', 'Nombre ratings', 'Sparsité user-item'],
    'Valeur': [n_users, n_movies, n_ratings, f'{sparsity:.2%}']
})

## 4) Distribution des notes

In [ ]:
ratings_pd = ratings_clean.select('userId', 'movieId', 'rating', 'timestamp').toPandas()
ratings_pd['datetime'] = pd.to_datetime(ratings_pd['timestamp'], unit='s')

stats_notes = pd.DataFrame({
    'Stat': ['Moyenne', 'Médiane', 'Skewness'],
    'Valeur': [ratings_pd['rating'].mean(), ratings_pd['rating'].median(), ratings_pd['rating'].skew()]
})
stats_notes.round(3)

In [ ]:
fig, ax = plt.subplots()
sns.histplot(ratings_pd['rating'], bins=10, kde=True, color='#1f77b4', ax=ax)
ax.set_title('Histogramme des notes')
ax.set_xlabel('Note')
ax.set_ylabel('Nombre')
plt.show()

## 5) Comportement utilisateurs

In [ ]:
ratings_per_user = ratings_pd.groupby('userId').size().rename('n_ratings')
ratings_per_user.describe().to_frame('valeur')

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))
sns.histplot(ratings_per_user, bins=40, color='#2ca02c', ax=axes[0])
axes[0].set_title('Ratings par user')
axes[0].set_xlabel('Nombre de ratings')
axes[0].set_ylabel('Nombre de users')

sns.histplot(ratings_per_user, bins=40, color='#ff7f0e', ax=axes[1])
axes[1].set_xscale('log')
axes[1].set_title('Ratings par user (log)')
axes[1].set_xlabel('Nombre de ratings (log)')
axes[1].set_ylabel('Nombre de users')

plt.tight_layout()
plt.show()

In [ ]:
p99 = ratings_per_user.quantile(0.99)
power_users = ratings_per_user[ratings_per_user >= p99]
pd.DataFrame({
    'Mesure': ['Seuil P99', 'Nombre power users', 'Part des users'],
    'Valeur': [int(p99), power_users.shape[0], f'{power_users.shape[0] / n_users:.2%}']
})

## 6) Popularité des films

In [ ]:
movies_pd = movies_clean.select('movieId', 'title', 'genres').toPandas()
ratings_per_movie = ratings_pd.groupby('movieId').size().rename('n_ratings')
movie_pop = movies_pd.merge(ratings_per_movie, on='movieId', how='left').fillna({'n_ratings': 0})
movie_pop['n_ratings'] = movie_pop['n_ratings'].astype(int)
top10 = movie_pop.sort_values('n_ratings', ascending=False).head(10)
top10[['title', 'n_ratings']]

In [ ]:
fig, ax = plt.subplots(figsize=(12, 7))
sns.barplot(data=top10, y='title', x='n_ratings', palette='Blues_r', ax=ax)
ax.set_title('Top 10 films les plus notés')
ax.set_xlabel('Nombre de ratings')
ax.set_ylabel('Titre')
plt.tight_layout()
plt.show()

In [ ]:
movie_counts_sorted = ratings_per_movie.sort_values(ascending=False).reset_index(drop=True)
cum_share = movie_counts_sorted.cumsum() / movie_counts_sorted.sum()
top20_n = int(round(0.2 * len(movie_counts_sorted)))
share_top20 = movie_counts_sorted.iloc[:top20_n].sum() / movie_counts_sorted.sum()

fig, ax = plt.subplots()
ax.plot(range(1, len(cum_share) + 1), cum_share, color='#d62728', linewidth=2)
ax.axvline(top20_n, color='gray', linestyle='--', label='Top 20% films')
ax.axhline(share_top20, color='black', linestyle=':', label=f'Part ratings top 20%: {share_top20:.1%}')
ax.set_title('Long tail popularité')
ax.set_xlabel('Films triés')
ax.set_ylabel('Part cumulée des ratings')
ax.legend()
plt.tight_layout()
plt.show()

## 7) Analyse temporelle

In [ ]:
monthly = ratings_pd.set_index('datetime').resample('M').size().rename('n_ratings')
yearly = ratings_pd.set_index('datetime').resample('Y').size().rename('n_ratings')

fig, axes = plt.subplots(2, 1, figsize=(14, 10))
monthly.plot(ax=axes[0], color='#1f77b4', linewidth=2)
axes[0].set_title('Ratings par mois')
axes[0].set_ylabel('Nombre')

yearly.plot(kind='bar', ax=axes[1], color='#17becf')
axes[1].set_title('Ratings par année')
axes[1].set_xlabel('Année')
axes[1].set_ylabel('Nombre')

plt.tight_layout()
plt.show()

In [ ]:
ratings_pd['month'] = ratings_pd['datetime'].dt.month
seasonality = ratings_pd.groupby('month').size().reindex(range(1, 13), fill_value=0)

fig, ax = plt.subplots()
sns.lineplot(x=seasonality.index, y=seasonality.values, marker='o', color='#9467bd', ax=ax)
ax.set_title('Saisonnalité simple')
ax.set_xlabel('Mois')
ax.set_ylabel('Nombre')
ax.set_xticks(range(1, 13))
plt.show()

## 8) Analyse des genres

In [ ]:
genres_pd = movie_genres.select('movieId', 'genre').toPandas()
ratings_genre = ratings_pd.merge(genres_pd, on='movieId', how='left')
genre_pop = ratings_genre.groupby('genre').size().sort_values(ascending=False).rename('n_ratings')
genre_avg = ratings_genre.groupby('genre')['rating'].mean().sort_values(ascending=False).rename('avg_rating')
genre_summary = pd.concat([genre_pop, genre_avg], axis=1).sort_values('n_ratings', ascending=False)
genre_summary.head(12)

In [ ]:
top_genres = genre_summary.head(12).reset_index()
top_avg_genres = genre_summary.sort_values('avg_rating', ascending=False).head(12).reset_index()

fig, axes = plt.subplots(1, 2, figsize=(18, 7))
sns.barplot(data=top_genres, y='genre', x='n_ratings', palette='Greens_r', ax=axes[0])
axes[0].set_title('Genres les plus notés')
axes[0].set_xlabel('Nombre de ratings')
axes[0].set_ylabel('Genre')

sns.barplot(data=top_avg_genres, y='genre', x='avg_rating', palette='Oranges_r', ax=axes[1])
axes[1].set_title('Genres avec note moyenne élevée')
axes[1].set_xlabel('Note moyenne')
axes[1].set_ylabel('Genre')

plt.tight_layout()
plt.show()

## 9) Démo modeling avec modules existants

In [ ]:
_, best_params = train_als_with_tuning(train_df, val_df, settings=settings)
als_model = retrain_best_als(train_df, val_df, best_params, settings=settings)
pred_test = score_als(als_model, test_df)

rmse = compute_rmse(pred_test)
mae = compute_mae(pred_test)

pd.DataFrame({'Metric': ['RMSE', 'MAE'], 'Value': [rmse, mae]})

In [ ]:
als_candidates = recommend_for_all_users_flat(als_model, 50).cache()
user_genre_profiles = build_user_genre_profiles(train_df, movie_genres).cache()
seen_interactions = train_df.select('userId', 'movieId').unionByName(val_df.select('userId', 'movieId')).cache()
content_candidates = generate_content_candidates(user_genre_profiles, movie_genre_weights, seen_interactions, k=50).cache()

user_tag_profiles = build_user_tag_profiles(tags_clean).cache()
movie_tag_features = build_tag_features(tags_clean).cache()

merged_candidates = (
    als_candidates.select('userId', 'movieId')
    .unionByName(content_candidates.select('userId', 'movieId'))
    .dropDuplicates(['userId', 'movieId'])
    .cache()
)

content_scores = build_content_scores(
    candidate_items_df=merged_candidates,
    user_genre_profiles_df=user_genre_profiles,
    movie_genre_weights_df=movie_genre_weights,
    user_tag_profiles_df=user_tag_profiles,
    movie_tag_features_df=movie_tag_features
).cache()

als_scores = score_als_candidates(als_model, merged_candidates).cache()
hybrid_scores = combine_hybrid_scores(als_scores, content_scores, settings=settings).cache()
top_recs = select_top_k_recommendations(hybrid_scores, settings.hybrid.top_k).cache()

precision_k = compute_precision_at_k(
    recommendations_df=top_recs.select('userId', 'movieId', 'rank', 'final_score'),
    ground_truth_df=test_df,
    k=settings.hybrid.top_k,
    positive_threshold=4.0
)

pd.DataFrame({'Metric': [f'Precision@{settings.hybrid.top_k}'], 'Value': [precision_k]})

In [ ]:
recs_demo = (
    top_recs.join(movies_clean.select('movieId', 'title', 'genres'), on='movieId', how='left')
    .select('userId', 'movieId', 'title', 'genres', 'rank', 'final_score', 'explanation')
)
recs_demo.orderBy('userId', 'rank').show(20, truncate=False)

## 10) Insights simples

- La matrice est sparse, donc ALS est un bon choix pour apprendre les préférences latentes.
- Le biais popularité existe, donc un hybride ALS + contenu est utile.
- Le cold-start reste un problème pour nouveaux users et nouveaux films.

## 11) Option pipeline complet

In [ ]:
# Optionnel: lance le pipeline de bout en bout
# result = run_pipeline(spark=spark, settings=settings, use_tags=True)
# result

## 12) Arrêt Spark

In [ ]:
# Décommentez quand vous avez fini
# spark.stop()